<a href="https://colab.research.google.com/github/MarcGaac/DSC1107/blob/main/LabFA_2_DSC1107.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
library(tidyverse)
data("starwars", package = "dplyr")

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


**Q1**

In [ ]:
# Structure of the dataset
glimpse(starwars)

# Dimensions
dim(starwars)

# Missing values in selected columns
starwars %>%
  summarise(
    na_height    = sum(is.na(height)),
    na_mass      = sum(is.na(mass)),
    na_homeworld = sum(is.na(homeworld))
  )

Rows: 87
Columns: 14
$ name       <chr> "Luke Skywalker", "C-3PO", "R2-D2", "Darth Vader", "Leia Or…
$ height     <int> 172, 167, 96, 202, 150, 178, 165, 97, 183, 182, 188, 180, 2…
$ mass       <dbl> 77.0, 75.0, 32.0, 136.0, 49.0, 120.0, 75.0, 32.0, 84.0, 77.…
$ hair_color <chr> "blond", NA, NA, "none", "brown", "brown, grey", "brown", N…
$ skin_color <chr> "fair", "gold", "white, blue", "white", "light", "light", "…
$ eye_color  <chr> "blue", "yellow", "red", "yellow", "brown", "blue", "blue",…
$ birth_year <dbl> 19.0, 112.0, 33.0, 41.9, 19.0, 52.0, 47.0, NA, 24.0, 57.0, …
$ sex        <chr> "male", "none", "none", "male", "female", "male", "female",…
$ gender     <chr> "masculine", "masculine", "masculine", "masculine", "femini…
$ homeworld  <chr> "Tatooine", "Tatooine", "Naboo", "Tatooine", "Alderaan", "T…
$ species    <chr> "Human", "Droid", "Droid", "Human", "Human", "Human", "Huma…
$ films      <list> <"A New Hope", "The Empire Strikes Back", "Return of the J…
$ vehicles   <list>

[1] 87 14

na_height,na_mass,na_homeworld
<int>,<int>,<int>
6,28,10


**Q2**

In [ ]:
wide_height <- starwars %>%
  filter(!is.na(species)) %>%                 # non‑missing species
  group_by(species, gender) %>%
  summarise(mean_height = mean(height, na.rm = TRUE), .groups = "drop") %>%
  pivot_wider(
    names_from  = gender,
    values_from = mean_height
  )

wide_height

species,masculine,feminine
<chr>,<dbl>,<dbl>
Aleena,79.0000,NA
Besalisk,198.0000,NA
Cerean,198.0000,NA
Chagrian,196.0000,NA
Clawdite,NA,168.0000
Droid,140.0000,96.0000
Dug,112.0000,NA
Ewok,88.0000,NA
Geonosian,183.0000,NA


**Q3**

In [ ]:
long_height <- wide_height %>%
  pivot_longer(
    cols      = -species,
    names_to  = "gender",
    values_to = "mean_height"
  ) %>%
  filter(!is.na(mean_height))   # remove missing mean heights

long_height

species,gender,mean_height
<chr>,<chr>,<dbl>
Aleena,masculine,79.0000
Besalisk,masculine,198.0000
Cerean,masculine,198.0000
Chagrian,masculine,196.0000
Clawdite,feminine,168.0000
Droid,masculine,140.0000
Droid,feminine,96.0000
Dug,masculine,112.0000
Ewok,masculine,88.0000


**Q4**

In [ ]:
starwars_extended <- starwars %>%
  mutate(
    bmi = mass / (height / 100)^2,
    height_cat = case_when(
      height < 170            ~ "short",
      height >= 170 & height <= 189 ~ "average",
      height >= 190           ~ "tall",
      TRUE                    ~ NA_character_
    ),
    homeworld = replace_na(homeworld, "Unknown")
  )

glimpse(starwars_extended)

Rows: 87
Columns: 16
$ name       <chr> "Luke Skywalker", "C-3PO", "R2-D2", "Darth Vader", "Leia Or…
$ height     <int> 172, 167, 96, 202, 150, 178, 165, 97, 183, 182, 188, 180, 2…
$ mass       <dbl> 77.0, 75.0, 32.0, 136.0, 49.0, 120.0, 75.0, 32.0, 84.0, 77.…
$ hair_color <chr> "blond", NA, NA, "none", "brown", "brown, grey", "brown", N…
$ skin_color <chr> "fair", "gold", "white, blue", "white", "light", "light", "…
$ eye_color  <chr> "blue", "yellow", "red", "yellow", "brown", "blue", "blue",…
$ birth_year <dbl> 19.0, 112.0, 33.0, 41.9, 19.0, 52.0, 47.0, NA, 24.0, 57.0, …
$ sex        <chr> "male", "none", "none", "male", "female", "male", "female",…
$ gender     <chr> "masculine", "masculine", "masculine", "masculine", "femini…
$ homeworld  <chr> "Tatooine", "Tatooine", "Naboo", "Tatooine", "Alderaan", "T…
$ species    <chr> "Human", "Droid", "Droid", "Human", "Human", "Human", "Huma…
$ films      <list> <"A New Hope", "The Empire Strikes Back", "Return of the J…
$ vehicles   <list>

**Q5**

In [ ]:
film_appearances <- starwars %>%
  select(name, films) %>%
  unnest(films) %>%                    # one row per film appearance
  count(name, sort = TRUE) %>%
  slice_max(n, n = 8)                  # top 8

film_appearances

name,n
<chr>,<int>
R2-D2,7
C-3PO,6
Obi-Wan Kenobi,6
Chewbacca,5
Leia Organa,5
Luke Skywalker,5
Palpatine,5
Yoda,5


The column representing films contains a cell for every film title which is stored as a list-column with one or more titles in each cell. We need these films listed in separate rows so that we can use aggregation functions such as count() to sum the number of of times a character appears across films. The result will be a tidy dataset with one observation of a characters appearance in a film per row thus allowing for easier calculation of group-wise summaries.